# ACLIF — Execution Notebook
## Autonomous Cross-Layer Intelligence Framework
**Author:** B. Tharuni Sri Sai | VIT-AP University, Amaravati, India  
**ORCID:** 0009-0004-9561-8985  
**Paper:** *Performance Optimization through Cross-Layer Intelligence in IoT Systems*

This notebook contains:
1. NS3 simulation configuration and launch
2. DQN agent training (PyTorch)
3. Cross-layer state aggregation (ICSA)
4. Multi-objective reward computation
5. Adaptive Protocol Reconfiguration (APR) logic

In [ ]:
# ── 0. Install dependencies ─────────────────────────────────────────────────
!pip install torch numpy pandas matplotlib scipy tqdm --quiet

In [ ]:
# ── 1. Imports ──────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import random
import os
import subprocess
import json
from collections import deque
from tqdm import tqdm

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)

print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

In [ ]:
# ── 2. ACLIF Simulation Parameters ──────────────────────────────────────────
PARAMS = {
    'N_NODES': 100,
    'AREA_SIDE': 200.0,          # metres
    'SIM_TIME': 300.0,           # seconds
    'CTRL_INTERVAL': 0.5,        # seconds
    'PKT_SIZE_BYTES': 512,
    'PKT_RATE': 5.0,             # pkt/s
    'INIT_ENERGY': 2.0,          # Joules
    'E_ELEC': 50e-9,             # J/bit
    'E_AMP': 1.3e-15,            # J/bit/m^alpha
    'PATH_LOSS_EXP': 3.5,
    'NOISE_PSD_DBM': -174,
    'W_MIN': 16,
    'W_MAX': 256,
    'BASE_TX_POWER_DBM': 0.0,
    'POWER_STEP_DBM': 3.0,
    'BASE_RATE_KBPS': 50.0,
    'RATE_SCALE': 1.5,
    'CLUSTER_UPDATE_INTERVAL': 50,
    # DQN
    'STATE_DIM': 15,
    'ACTION_DIM': 81,            # 3^4
    'HIDDEN1': 128,
    'HIDDEN2': 64,
    'LR': 1e-3,
    'GAMMA': 0.95,
    'EPSILON_MAX': 1.0,
    'EPSILON_MIN': 0.1,
    'EPSILON_DECAY': 0.01,
    'BATCH_SIZE': 64,
    'REPLAY_SIZE': 1000,
    'TARGET_UPDATE_FREQ': 100,
    'N_EPISODES': 300,
    # Reward weights
    'W1': 0.30,  # throughput
    'W2': 0.25,  # delay
    'W3': 0.20,  # energy
    'W4': 0.15,  # collision
    'W5': 0.10,  # overhead
}
print(json.dumps(PARAMS, indent=2))

In [ ]:
# ── 3. DQN Network Architecture ─────────────────────────────────────────────
class DQN(nn.Module):
    """Deep Q-Network: state_dim → 128 → 64 → action_dim (81)"""
    def __init__(self, state_dim, action_dim, hidden1=128, hidden2=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden1),
            nn.ReLU(),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Linear(hidden2, action_dim)
        )
    
    def forward(self, x):
        return self.net(x)

# Instantiate policy and target networks
policy_net = DQN(PARAMS['STATE_DIM'], PARAMS['ACTION_DIM'],
                 PARAMS['HIDDEN1'], PARAMS['HIDDEN2']).to(DEVICE)
target_net = DQN(PARAMS['STATE_DIM'], PARAMS['ACTION_DIM'],
                 PARAMS['HIDDEN1'], PARAMS['HIDDEN2']).to(DEVICE)
target_net.load_state_dict(policy_net.state_dict())
target_net.eval()

optimizer = optim.Adam(policy_net.parameters(), lr=PARAMS['LR'])
criterion = nn.MSELoss()

print(policy_net)
total_params = sum(p.numel() for p in policy_net.parameters())
print(f'Total parameters: {total_params:,}')

In [ ]:
# ── 4. Action Space: encode/decode 81 joint actions ─────────────────────────
def build_action_space():
    """Build all 81 = 3^4 joint actions.
    Each action is (pwr_delta, win_delta, route_mode, rate_delta)
    Each component ∈ {-1, 0, +1}
    """
    actions = []
    for pwr in [-1, 0, 1]:
        for win in [-1, 0, 1]:
            for route in [-1, 0, 1]:  # maps to modes 0,1,2
                for rate in [-1, 0, 1]:
                    actions.append((pwr, win, route + 1, rate))
    return actions

ACTION_SPACE = build_action_space()
print(f'Action space size: {len(ACTION_SPACE)}')
print('Example actions:')
for i in [0, 40, 80]:
    print(f'  [{i}] pwr={ACTION_SPACE[i][0]:+d} win={ACTION_SPACE[i][1]:+d} route={ACTION_SPACE[i][2]} rate={ACTION_SPACE[i][3]:+d}')

In [ ]:
# ── 5. ICSA: Cross-Layer State Aggregation ──────────────────────────────────
def icsa_aggregate(node_states, now, delta_max=1.0, tau_s=0.5):
    """
    Phase 1: Intelligent Cross-Layer State Aggregation
    
    node_states: list of dicts with keys:
      snr, tx_power, residual_energy, cont_win, collision_rate,
      queue_len, hop_count, e2e_delay, send_rate, rtt, comm_overhead,
      last_update_time, alive
    
    Returns: normalised 15-dim state vector as numpy array
    """
    valid = [n for n in node_states
             if n['alive'] and
             np.exp(-max(0, now - n['last_update_time'] - delta_max) / tau_s) >= 0.5]
    
    if not valid:
        return np.zeros(15, dtype=np.float32)
    
    snrs   = [n['snr'] for n in valid]
    raw = np.array([
        np.mean(snrs),                           # PHY.1 mean SNR
        np.var(snrs),                            # PHY.2 SNR variance
        np.mean([n['tx_power'] for n in valid]), # PHY.3 mean Tx power
        np.mean([n['residual_energy'] for n in valid]),  # PHY.4 mean energy
        np.mean([n['cont_win'] for n in valid]), # MAC.1 mean CW
        np.mean([n['collision_rate'] for n in valid]),   # MAC.2 mean collision
        np.mean([n['queue_len'] for n in valid]),# MAC.3 mean queue
        len(valid) / len(node_states),           # MAC.4 channel occupancy
        np.mean([n['hop_count'] for n in valid]),# NET.1 mean hops
        np.mean([n['e2e_delay'] for n in valid]),# NET.2 mean E2E delay
        sum(1 for n in node_states if n['alive']),  # NET.3 alive nodes
        np.mean([n.get('throughput', 0) for n in valid]),  # NET.4 throughput
        np.mean([n['send_rate'] for n in valid]),# TRP.1 send rate
        np.mean([n['rtt'] for n in valid]),      # TRP.2 RTT
        np.mean([n['comm_overhead'] for n in valid]),    # TRP.3 comm overhead
    ], dtype=np.float32)
    
    # Min-max normalisation (clip to [0,1])
    s_min = np.array([0, 0, -10, 0, 16, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0], dtype=np.float32)
    s_max = np.array([40, 100, 20, 2, 256, 1, 50, 1, 10, 1, 100, 100, 150, 1, 1], dtype=np.float32)
    normalised = np.clip((raw - s_min) / (s_max - s_min + 1e-8), 0, 1)
    return normalised

print('ICSA function ready.')

In [ ]:
# ── 6. Reward Function ──────────────────────────────────────────────────────
def compute_reward(throughput, delay, energy, collision, comm_overhead):
    """Multi-objective reward (Equation 15 in paper)"""
    w1, w2, w3, w4, w5 = PARAMS['W1'], PARAMS['W2'], PARAMS['W3'], PARAMS['W4'], PARAMS['W5']
    return w1*throughput - w2*delay - w3*energy - w4*collision - w5*comm_overhead

# Test reward function
r_test = compute_reward(throughput=83.7, delay=0.0915, energy=0.0291, collision=0.083, comm_overhead=0.087)
print(f'Sample ACLIF reward: {r_test:.4f}')

In [ ]:
# ── 7. Experience Replay Buffer ─────────────────────────────────────────────
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)
    
    def push(self, state, action_idx, reward, next_state, done):
        self.buffer.append((state, action_idx, reward, next_state, done))
    
    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            torch.FloatTensor(np.array(states)).to(DEVICE),
            torch.LongTensor(actions).to(DEVICE),
            torch.FloatTensor(rewards).to(DEVICE),
            torch.FloatTensor(np.array(next_states)).to(DEVICE),
            torch.BoolTensor(dones).to(DEVICE)
        )
    
    def __len__(self):
        return len(self.buffer)

replay_buffer = ReplayBuffer(PARAMS['REPLAY_SIZE'])
print('Replay buffer ready.')

In [ ]:
# ── 8. IoT Environment Simulator (NS3 surrogate) ────────────────────────────
class IoTEnvSimulator:
    """Lightweight Python surrogate of the NS3 IoT environment.
    Replace step() with NS3 subprocess call for full simulation."""
    
    def __init__(self, n_nodes=100, area=200.0):
        self.n_nodes = n_nodes
        self.area = area
        self.reset()
    
    def reset(self):
        np.random.seed(np.random.randint(10000))
        self.nodes = [{
            'pos': np.random.uniform(0, self.area, 2),
            'residual_energy': PARAMS['INIT_ENERGY'],
            'snr': np.random.uniform(5, 30),
            'tx_power': PARAMS['BASE_TX_POWER_DBM'],
            'cont_win': PARAMS['W_MIN'],
            'collision_rate': 0.05,
            'queue_len': 0,
            'hop_count': 2,
            'e2e_delay': 0.09,
            'send_rate': PARAMS['BASE_RATE_KBPS'],
            'rtt': 0.18,
            'comm_overhead': 0.09,
            'throughput': 50.0,
            'last_update_time': 0.0,
            'alive': True
        } for _ in range(n_nodes)]
        self.t = 0.0
        return icsa_aggregate(self.nodes, self.t)
    
    def step(self, action_idx):
        pwr, win, route, rate = ACTION_SPACE[action_idx]
        
        for n in self.nodes:
            if not n['alive']: continue
            # Apply action
            n['tx_power'] = PARAMS['BASE_TX_POWER_DBM'] + PARAMS['POWER_STEP_DBM'] * pwr
            if win == 1:  n['cont_win'] = min(n['cont_win']*2, PARAMS['W_MAX'])
            elif win==-1: n['cont_win'] = max(n['cont_win']//2, PARAMS['W_MIN'])
            if rate == 1:  n['send_rate'] = PARAMS['BASE_RATE_KBPS'] * PARAMS['RATE_SCALE']
            elif rate==-1: n['send_rate'] = PARAMS['BASE_RATE_KBPS'] / PARAMS['RATE_SCALE']
            # Simulate dynamics
            n['snr'] = np.clip(n['snr'] + pwr*2 + np.random.normal(0, 1.5), 0, 40)
            n['collision_rate'] = max(0.01, 1 - (1 - 1/n['cont_win'])**max(1, self.n_nodes//10 - 1))
            n['e2e_delay'] = np.clip(0.09 - pwr*0.003 - win*0.005 + np.random.normal(0,0.005), 0.03, 0.5)
            n['throughput'] = np.clip(n['send_rate']*0.9 - n['collision_rate']*20 + np.random.normal(0,2), 10, 120)
            n['rtt'] = n['e2e_delay'] * 2
            # Energy drain
            e_tx = PARAMS['PKT_SIZE_BYTES']*8 * (PARAMS['E_ELEC'] + PARAMS['E_AMP']*50**PARAMS['PATH_LOSS_EXP'])
            n['residual_energy'] -= e_tx * PARAMS['PKT_RATE'] * PARAMS['CTRL_INTERVAL']
            if n['residual_energy'] <= 0: n['alive'] = False
            n['last_update_time'] = self.t
        
        self.t += PARAMS['CTRL_INTERVAL']
        alive = [n for n in self.nodes if n['alive']]
        
        throughput = np.mean([n['throughput'] for n in alive]) if alive else 0
        delay = np.mean([n['e2e_delay'] for n in alive]) if alive else 1
        energy = PARAMS['INIT_ENERGY'] - (np.mean([n['residual_energy'] for n in alive]) if alive else 0)
        collision = np.mean([n['collision_rate'] for n in alive]) if alive else 1
        comm_oh = np.mean([n['comm_overhead'] for n in alive]) if alive else 1
        
        reward = compute_reward(throughput/100, delay, energy/2, collision, comm_oh)
        done = len(alive) < self.n_nodes * 0.1 or self.t >= PARAMS['SIM_TIME']
        next_state = icsa_aggregate(self.nodes, self.t)
        
        info = {'throughput': throughput, 'delay': delay*1000,
                'energy': energy, 'collision': collision*100, 'alive': len(alive)}
        return next_state, reward, done, info

env = IoTEnvSimulator(PARAMS['N_NODES'])
s0 = env.reset()
print(f'Initial state shape: {s0.shape}')
print(f'State sample: {s0}')

In [ ]:
# ── 9. DQN Training Loop ─────────────────────────────────────────────────────
def train_dqn(env, n_episodes=300):
    episode_rewards = []
    episode_delays  = []
    episode_tputs   = []
    epsilon         = PARAMS['EPSILON_MAX']
    step_count      = 0
    
    for ep in tqdm(range(n_episodes), desc='Training DQN'):
        state = env.reset()
        ep_reward = 0
        ep_delay  = []
        ep_tput   = []
        done = False
        
        while not done:
            # ε-greedy action selection
            if random.random() < epsilon:
                action_idx = random.randrange(PARAMS['ACTION_DIM'])
            else:
                with torch.no_grad():
                    q_vals = policy_net(torch.FloatTensor(state).unsqueeze(0).to(DEVICE))
                    action_idx = q_vals.argmax().item()
            
            next_state, reward, done, info = env.step(action_idx)
            replay_buffer.push(state, action_idx, reward, next_state, done)
            state = next_state
            ep_reward += reward
            ep_delay.append(info['delay'])
            ep_tput.append(info['throughput'])
            step_count += 1
            
            # Train if buffer has enough samples
            if len(replay_buffer) >= PARAMS['BATCH_SIZE']:
                states_b, actions_b, rewards_b, next_states_b, dones_b = \
                    replay_buffer.sample(PARAMS['BATCH_SIZE'])
                
                # Bellman target
                with torch.no_grad():
                    max_q_next = target_net(next_states_b).max(1)[0]
                    target_q   = rewards_b + PARAMS['GAMMA'] * max_q_next * (~dones_b)
                
                current_q = policy_net(states_b).gather(1, actions_b.unsqueeze(1)).squeeze(1)
                loss = criterion(current_q, target_q)
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(policy_net.parameters(), 1.0)
                optimizer.step()
            
            # Sync target network
            if step_count % PARAMS['TARGET_UPDATE_FREQ'] == 0:
                target_net.load_state_dict(policy_net.state_dict())
        
        # Decay epsilon
        epsilon = max(PARAMS['EPSILON_MIN'],
                      PARAMS['EPSILON_MIN'] + (PARAMS['EPSILON_MAX'] - PARAMS['EPSILON_MIN']) *
                      np.exp(-PARAMS['EPSILON_DECAY'] * ep))
        
        episode_rewards.append(ep_reward)
        episode_delays.append(np.mean(ep_delay) if ep_delay else 0)
        episode_tputs.append(np.mean(ep_tput) if ep_tput else 0)
    
    return episode_rewards, episode_delays, episode_tputs

print('Starting DQN training for', PARAMS['N_EPISODES'], 'episodes...')
rewards_hist, delays_hist, tputs_hist = train_dqn(env, PARAMS['N_EPISODES'])
print('Training complete!')

In [ ]:
# ── 10. Save trained model ──────────────────────────────────────────────────
os.makedirs('models', exist_ok=True)
torch.save({
    'policy_state_dict': policy_net.state_dict(),
    'target_state_dict': target_net.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'params': PARAMS
}, 'models/aclif_dqn.pth')
print('Model saved to models/aclif_dqn.pth')

In [ ]:
# ── 11. Training convergence plot ───────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
window = 15
eps = np.arange(1, len(rewards_hist)+1)

for ax, data, label, color in zip(axes,
    [rewards_hist, delays_hist, tputs_hist],
    ['Cumulative Reward', 'Avg Delay (ms)', 'Avg Throughput (kbps)'],
    ['#6366f1', '#e63946', '#2a9d8f']):
    ax.plot(eps, data, alpha=0.3, color=color, lw=0.8)
    ma = np.convolve(data, np.ones(window)/window, mode='valid')
    ax.plot(np.arange(window, len(data)+1), ma, color=color, lw=2)
    ax.set_xlabel('Episode'); ax.set_ylabel(label)
    ax.set_title(label + ' vs Episode')
    ax.grid(alpha=0.3)

plt.suptitle('ACLIF DQN Training Convergence', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/training_convergence.png', dpi=150)
plt.show()
print('Convergence plot saved.')

In [ ]:
# ── 12. NS3 Subprocess Launcher (production use) ────────────────────────────
def run_ns3_simulation(script_name, args_dict, ns3_path='~/ns-3.38'):
    """
    Launch an NS3 simulation script via subprocess.
    Replace the IoTEnvSimulator above with this for full NS3 integration.
    
    Args:
        script_name: e.g. 'res1' (must exist in ns3_path/scratch/)
        args_dict: dict of NS3 command-line args
        ns3_path: path to NS3 installation
    """
    args_str = ' '.join([f'--{k}={v}' for k, v in args_dict.items()])
    cmd = f'cd {ns3_path} && ./ns3 run "scratch/{script_name} {args_str}"'
    print(f'Running: {cmd}')
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print('STDERR:', result.stderr[:500])
    return result.stdout

# Example (commented out — requires NS3 installed):
# output = run_ns3_simulation('res1', {'nodeCount': 100, 'seed': 42})
# print(output)
print('NS3 launcher function defined.')

In [ ]:
# ── 13. Summary ─────────────────────────────────────────────────────────────
print('=' * 60)
print('ACLIF Execution Summary')
print('=' * 60)
print(f'DQN converged at episode ~ {np.argmax(np.convolve(rewards_hist, np.ones(20)/20, mode="valid")[-50:]) + len(rewards_hist) - 50}')
print(f'Final avg reward (last 20 eps): {np.mean(rewards_hist[-20:]):.4f}')
print(f'Final avg delay  (last 20 eps): {np.mean(delays_hist[-20:]):.2f} ms')
print(f'Final avg tput   (last 20 eps): {np.mean(tputs_hist[-20:]):.2f} kbps')
print(f'Model saved: models/aclif_dqn.pth')
print('=' * 60)